In [1]:
!pip install albumentations opencv-python tqdm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 46.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 82.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 48.5 MB/s eta 0:00:00
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 582.3/582.3 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.0 MB/s eta 0:00:00a 0:00:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Consider adding this directory to PATH or, i

In [1]:
import os
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key="kHzDvVLoFaxNmCTjPHiX")
project = rf.workspace("lookie").project("my-first-project-sudpc")
version = project.version(5)
dataset = version.download("yolov8")
print(f"✅ 데이터셋 다운로드 완료 경로: {dataset.location}")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to My-First-Project-5 in yolov8:: 100%|██████████| 382/382 [00:00<00:00, 20246.98it/s]

✅ 데이터셋 다운로드 완료 경로: /home/j-i14b105/My-First-Project-5


In [8]:
import albumentations as A
import cv2
import os
import glob
from tqdm import tqdm

# ================= 설정 구간 =================
INPUT_IMG_DIR = 'My-First-Project-4/train/images'
INPUT_LABEL_DIR = 'My-First-Project-4/train/labels'

OUTPUT_IMG_DIR = 'My-First-Project-4/aug_train/images'
OUTPUT_LABEL_DIR = 'My-First-Project-4/aug_train/labels'

AUG_FACTOR = 10
# ============================================

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

# 🔥 YOLO-safe augmentation pipeline
transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Rotate(
            limit=15,
            p=0.5,
            border_mode=cv2.BORDER_CONSTANT,
            value=0
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.3,
            contrast_limit=0.3,
            p=0.5
        ),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        min_visibility=0.3,
        label_fields=['class_labels']
    )
)

image_paths = glob.glob(os.path.join(INPUT_IMG_DIR, '*.jpg'))
print(f"🚀 {len(image_paths)}장의 이미지를 {AUG_FACTOR}배 증강합니다.")

success_count = 0

for img_path in tqdm(image_paths):
    filename = os.path.basename(img_path)
    name, ext = os.path.splitext(filename)
    label_path = os.path.join(INPUT_LABEL_DIR, name + '.txt')

    # 이미지 로드
    image = cv2.imread(img_path)
    if image is None:
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 라벨 파일 없으면 skip
    if not os.path.exists(label_path):
        continue

    if os.path.getsize(label_path) == 0:
        continue

    bboxes = []
    class_labels = []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls = int(parts[0])
                bbox = list(map(float, parts[1:]))

                # YOLO 좌표 유효성 검사
                if all(0.0 <= v <= 1.0 for v in bbox):
                    bboxes.append(bbox)
                    class_labels.append(cls)

    if len(bboxes) == 0:
        continue

    # 증강
    for i in range(AUG_FACTOR):
        try:
            augmented = transform(
                image=image,
                bboxes=bboxes,
                class_labels=class_labels
            )

            aug_img = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_labels = augmented['class_labels']  # ⭐ 핵심

            if len(aug_bboxes) == 0:
                continue

            save_name = f"{name}_aug_{i}"

            # 이미지 저장
            save_img = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(
                os.path.join(OUTPUT_IMG_DIR, save_name + ext),
                save_img
            )

            # 라벨 저장
            with open(os.path.join(OUTPUT_LABEL_DIR, save_name + '.txt'), 'w') as f:
                for bbox, cls in zip(aug_bboxes, aug_labels):
                    x, y, w, h = bbox
                    f.write(
                        f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"
                    )

            success_count += 1

        except Exception:
            continue

print(f"✅ 완료: 총 {success_count}개 증강 데이터 생성")
print(f"📂 저장 위치: {OUTPUT_IMG_DIR}")

print("\n🔍 bbox 검증 이미지 생성 중...")

sample_files = os.listdir(OUTPUT_IMG_DIR)
if not sample_files:
    print("❌ 생성된 이미지 없음")
    exit()

sample_file = sample_files[0]
name, ext = os.path.splitext(sample_file)

img = cv2.imread(os.path.join(OUTPUT_IMG_DIR, sample_file))
h, w, _ = img.shape

label_path = os.path.join(OUTPUT_LABEL_DIR, name + '.txt')

with open(label_path, 'r') as f:
    for line in f:
        cls, cx, cy, bw, bh = map(float, line.split())
        x1 = int((cx - bw / 2) * w)
        y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w)
        y2 = int((cy + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

cv2.imwrite("check_augmentation.jpg", img)
print("📸 check_augmentation.jpg 생성 완료")


Argument(s) 'value' are not valid for transform Rotate


🚀 54장의 이미지를 10배 증강합니다.


100%|██████████| 54/54 [00:01<00:00, 48.46it/s]

✅ 완료: 총 300개 증강 데이터 생성
📂 저장 위치: My-First-Project-4/aug_train/images

🔍 bbox 검증 이미지 생성 중...
📸 check_augmentation.jpg 생성 완료


In [2]:
# ---------------------------------------------------------
# Step 2. 모델 학습 (Training)
# ---------------------------------------------------------
# CPU 추론용으로 가장 가볍고 빠른 'Nano' 모델(yolov8n.pt) 사용
model = YOLO('yolov8n.pt') 

print("🚀 학습을 시작합니다! (예상 소요 시간: 15~30분)")

results = model.train(
    data=f"{dataset.location}/data.yaml",  # 다운받은 데이터셋 경로 자동 연결
    epochs=300,                            # 학습 반복 횟수 (데이터가 적어서 금방 끝남)
    imgsz=640,                             # 아까 설정한 640 사이즈
    patience=100,                           # 20번 동안 성능 안 오르면 조기 종료 (시간 절약)
    batch=16,                              # 메모리 부족 에러 나면 8로 줄일 것
    device=0,                              # GPU 0번 사용 (필수)
    project='SFS_Logistics',               # 프로젝트 폴더명
    name='banana_detect_v3(epoch=300,patience=100)'                # 이번 실험 이름
)

🚀 학습을 시작합니다! (예상 소요 시간: 15~30분)
New https://pypi.org/project/ultralytics/8.4.8 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.10.10 torch-2.9.1+cu128 CUDA:0 (NVIDIA L40S, 45596MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/j-i14b105/My-First-Project-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, mult

In [7]:
!rm -rf My-First-Project-4/aug_train/